# Project 9: Student Degree Classification (TensorFlow - High Accuracy)

Build a high-accuracy neural network using TensorFlow/Keras to classify students into degree categories based on their academic performance.

## Objectives
- Load and preprocess CSV data with advanced techniques
- Build a deep neural network using TensorFlow/Keras
- Implement class balancing for imbalanced data
- Use TensorFlow callbacks (early stopping, learning rate scheduling)
- Achieve high accuracy (>90%) through TensorFlow optimizations
- Evaluate model with comprehensive metrics
- Compare TensorFlow performance with custom implementations

## Key Features Using TensorFlow

1. **TensorFlow/Keras**: Industry-standard deep learning framework
2. **Deeper Architecture**: 7 → 128 → 256 → 128 → 64 → 5 with dropout
3. **Class Balancing**: SMOTE-like oversampling + class weights
4. **Advanced Callbacks**: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
5. **Batch Normalization**: For stable training
6. **Dropout Regularization**: Prevent overfitting
7. **Feature Engineering**: Polynomial features and interaction terms
8. **Optimized Training**: Adam optimizer with learning rate scheduling

## Degree Categories

Based on final score:
- **Bad**: score < 50
- **Acceptable**: 50 ≤ score < 65
- **Good**: 65 ≤ score < 75
- **Very Good**: 75 ≤ score < 85
- **Excellent**: score ≥ 85

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (classification_report, confusion_matrix, 
                           accuracy_score, f1_score, precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check GPU availability
print(f"\nGPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"TensorFlow built with CUDA: {tf.test.is_built_with_cuda()}")

## Step 1: Load and Explore Dataset

In [ ]:
# Load dataset from CSV
print("Loading student dataset...")
df = pd.read_csv('data/neural_networks/student_degree_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nDataset Statistics:")
print(df.describe())

print(f"\nDegree Category Distribution:")
category_counts = df['degree_category'].value_counts().sort_index()
print(category_counts)

# Visualize class distribution
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
category_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Degree Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
category_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90, 
                    colors=sns.color_palette("husl", len(category_counts)))
plt.title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.tight_layout()
plt.show()

print(f"\nClass Imbalance Ratio: {category_counts.max() / category_counts.min():.2f}x")

## Step 2: Advanced Data Preprocessing with Feature Engineering

In [ ]:
# Prepare features and labels
feature_columns = ['attendance', 'quiz_avg', 'assignment_avg', 'midterm_score',
                   'project_score', 'study_hours_per_week', 'participation_score']
X = df[feature_columns].values
y_categories = df['degree_category'].values

# Map categories to numbers
category_mapping = {
    'Bad': 0,
    'Acceptable': 1,
    'Good': 2,
    'Very Good': 3,
    'Excellent': 4
}
y = np.array([category_mapping[cat] for cat in y_categories])

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")

# Feature Engineering: Add polynomial features (degree 2, interaction only)
print("\nCreating polynomial features...")
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_poly = poly.fit_transform(X)
print(f"Original features: {X.shape[1]}")
print(f"Polynomial features: {X_poly.shape[1]}")

X_enhanced = X_poly

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_enhanced, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {len(np.unique(y))}")

In [ ]:
# Advanced Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")
print(f"Training mean: {X_train_scaled.mean(axis=0)[:5]}")
print(f"Training std: {X_train_scaled.std(axis=0)[:5]}")

# One-hot encode labels for TensorFlow
num_classes = 5
y_train_categorical = to_categorical(y_train, num_classes)
y_test_categorical = to_categorical(y_test, num_classes)

print(f"\nOne-hot encoded labels shape: {y_train_categorical.shape}")

# Compute class weights for imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights for balancing: {class_weight_dict}")

# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(feature_columns):
    if idx < len(axes):
        axes[idx].hist(df[col], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        axes[idx].set_xlabel(col, fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
        axes[idx].grid(True, alpha=0.3)

axes[7].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Class Balancing with Oversampling

In [ ]:
# Simple oversampling for minority classes
def oversample_minority_classes(X, y, y_categorical, target_samples=350):
    """Oversample minority classes to balance the dataset"""
    unique_classes, class_counts = np.unique(y, return_counts=True)
    
    X_balanced = [X]
    y_balanced = [y]
    y_categorical_balanced = [y_categorical]
    
    for class_idx, count in zip(unique_classes, class_counts):
        if count < target_samples:
            # Find samples of this class
            class_mask = y == class_idx
            X_class = X[class_mask]
            y_class = y[class_mask]
            y_categorical_class = y_categorical[class_mask]
            
            # Calculate how many samples to add
            n_samples_needed = target_samples - count
            
            # Random oversampling with replacement
            indices = np.random.choice(len(X_class), size=n_samples_needed, replace=True)
            X_oversampled = X_class[indices]
            y_oversampled = y_class[indices]
            y_categorical_oversampled = y_categorical_class[indices]
            
            X_balanced.append(X_oversampled)
            y_balanced.append(y_oversampled)
            y_categorical_balanced.append(y_categorical_oversampled)
    
    X_balanced = np.vstack(X_balanced)
    y_balanced = np.hstack(y_balanced)
    y_categorical_balanced = np.vstack(y_categorical_balanced)
    
    # Shuffle
    shuffle_idx = np.random.permutation(len(X_balanced))
    X_balanced = X_balanced[shuffle_idx]
    y_balanced = y_balanced[shuffle_idx]
    y_categorical_balanced = y_categorical_balanced[shuffle_idx]
    
    return X_balanced, y_balanced, y_categorical_balanced

# Apply oversampling
print("Before oversampling:")
print(f"Class distribution: {np.bincount(y_train)}")

X_train_balanced, y_train_balanced, y_train_categorical_balanced = oversample_minority_classes(
    X_train_scaled, y_train, y_train_categorical, target_samples=300
)

print("\nAfter oversampling:")
print(f"Class distribution: {np.bincount(y_train_balanced)}")
print(f"Training samples: {len(X_train_balanced)}")

# Create validation set
X_train_final, X_val, y_train_final, y_val, y_train_final_cat, y_val_cat = train_test_split(
    X_train_balanced, y_train_balanced, y_train_categorical_balanced,
    test_size=0.15, random_state=42, stratify=y_train_balanced
)

print(f"\nFinal training set: {X_train_final.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

In [ ]:
# Build deep neural network using TensorFlow/Keras
input_size = X_train_final.shape[1]
print(f"Building TensorFlow/Keras model...")
print(f"Input size: {input_size}")
print(f"Architecture: {input_size} → 128 → 256 → 128 → 64 → 5")

model = models.Sequential([
    # Input layer
    layers.Dense(128, activation='relu', input_shape=(input_size,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 1
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    # Hidden layer 2
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 3
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Output layer
    layers.Dense(5, activation='softmax')
])

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_k_categorical_accuracy']
)

# Display model architecture
print("\n" + "=" * 60)
print("Model Architecture:")
print("=" * 60)
model.summary()

# Visualize model architecture
keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, 
                       to_file='docs/images/tensorflow_model_architecture.png', 
                       dpi=150)

## Step 5: Setup TensorFlow Callbacks

In [ ]:
# Setup TensorFlow callbacks for training optimization
callbacks_list = [
    # Early stopping to prevent overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=0.00001,
        verbose=1
    ),
    
    # Model checkpoint to save best model
    callbacks.ModelCheckpoint(
        'data/neural_networks/best_tensorflow_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # CSV logger for training history
    callbacks.CSVLogger(
        'data/neural_networks/training_history.csv',
        append=False
    )
]

print("TensorFlow callbacks configured:")
print("1. EarlyStopping: Stop if no improvement for 25 epochs")
print("2. ReduceLROnPlateau: Reduce LR by 0.5x if no improvement for 10 epochs")
print("3. ModelCheckpoint: Save best model based on validation accuracy")
print("4. CSVLogger: Log training history to CSV")

## Step 6: Train TensorFlow Model

In [ ]:
# Compute class weights for training
class_weights_final = compute_class_weight('balanced', 
                                          classes=np.unique(y_train_final), 
                                          y=y_train_final)
class_weight_dict_final = dict(enumerate(class_weights_final))

print("Training TensorFlow model...")
print("=" * 60)

# Train model
history = model.fit(
    X_train_final, y_train_final_cat,
    validation_data=(X_val, y_val_cat),
    epochs=300,
    batch_size=32,
    class_weight=class_weight_dict_final,
    callbacks=callbacks_list,
    verbose=1
)

print("\n" + "=" * 60)
print("Training complete!")
print("=" * 60)

# Load best model
model.load_weights('data/neural_networks/best_tensorflow_model.h5')
print("Best model weights loaded!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training and validation loss
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Training and validation accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning rate (if available)
if 'lr' in history.history:
    axes[1, 0].plot(history.history['lr'], linewidth=2, color='green')
    axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].axis('off')

# Top-k accuracy (if available)
if 'top_k_categorical_accuracy' in history.history:
    axes[1, 1].plot(history.history['top_k_categorical_accuracy'], 
                   label='Training Top-K', linewidth=2)
    axes[1, 1].plot(history.history['val_top_k_categorical_accuracy'], 
                   label='Validation Top-K', linewidth=2)
    axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Top-K Accuracy', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Top-K Categorical Accuracy', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")

## Step 7: Evaluate Model Performance

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_accuracy, test_top_k = model.evaluate(
    X_test_scaled, y_test_categorical, verbose=0
)

# Make predictions
predictions = model.predict(X_test_scaled, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test_categorical, axis=1)

# Calculate comprehensive metrics
accuracy = accuracy_score(actual_classes, predicted_classes)
f1_macro = f1_score(actual_classes, predicted_classes, average='macro')
f1_weighted = f1_score(actual_classes, predicted_classes, average='weighted')
precision = precision_score(actual_classes, predicted_classes, average='weighted')
recall = recall_score(actual_classes, predicted_classes, average='weighted')

print("\n" + "=" * 70)
print("TENSORFLOW MODEL EVALUATION RESULTS")
print("=" * 70)
print(f"\nTest Loss: {test_loss:.6f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Top-K Accuracy: {test_top_k:.4f} ({test_top_k*100:.2f}%)")
print(f"\nAdditional Metrics:")
print(f"  Accuracy (sklearn): {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  F1 Score (Macro): {f1_macro:.4f}")
print(f"  F1 Score (Weighted): {f1_weighted:.4f}")
print(f"  Precision (Weighted): {precision:.4f}")
print(f"  Recall (Weighted): {recall:.4f}")

# Classification report
category_names = ['Bad', 'Acceptable', 'Good', 'Very Good', 'Excellent']
print("\n" + "=" * 70)
print("Detailed Classification Report:")
print("=" * 70)
print(classification_report(actual_classes, predicted_classes, 
                          target_names=category_names, 
                          labels=[0, 1, 2, 3, 4],
                          zero_division=0))

# Confusion matrix
cm = confusion_matrix(actual_classes, predicted_classes)
print("\n" + "=" * 70)
print("Confusion Matrix:")
print("=" * 70)
print(cm)

# Save model info
model_info = {
    'framework': 'TensorFlow/Keras',
    'accuracy': float(test_accuracy),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'precision': float(precision),
    'recall': float(recall),
    'architecture': [input_size, 128, 256, 128, 64, 5],
    'training_samples': int(X_train_final.shape[0]),
    'validation_samples': int(X_val.shape[0]),
    'test_samples': int(X_test_scaled.shape[0]),
    'features': int(input_size),
    'best_val_accuracy': float(max(history.history['val_accuracy']))
}

import json
with open('data/neural_networks/tensorflow_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"\n[OK] Model info saved to: data/neural_networks/tensorflow_model_info.json")

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confusion Matrix
im = axes[0, 0].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0, 0].set_xticks(range(5))
axes[0, 0].set_yticks(range(5))
axes[0, 0].set_xticklabels(category_names, rotation=45, ha='right')
axes[0, 0].set_yticklabels(category_names)
axes[0, 0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[0, 0].text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0, 0])

# 2. Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = axes[0, 1].imshow(cm_normalized, cmap='Greens', interpolation='nearest', vmin=0, vmax=1)
axes[0, 1].set_xticks(range(5))
axes[0, 1].set_yticks(range(5))
axes[0, 1].set_xticklabels(category_names, rotation=45, ha='right')
axes[0, 1].set_yticklabels(category_names)
axes[0, 1].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[0, 1].text(j, i, f'{cm_normalized[i, j]:.2f}', ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if cm_normalized[i, j] > 0.5 else 'black')
plt.colorbar(im2, ax=axes[0, 1])

# 3. Per-class Accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
axes[1, 0].bar(category_names, per_class_accuracy, color='steelblue', edgecolor='black')
axes[1, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Per-Class Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(per_class_accuracy):
    axes[1, 0].text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# 4. Metrics Comparison
metrics = ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'Precision', 'Recall']
values = [accuracy, f1_macro, f1_weighted, precision, recall]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
axes[1, 1].barh(metrics, values, color=colors, edgecolor='black')
axes[1, 1].set_xlabel('Score', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Model Metrics', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim([0, 1])
axes[1, 1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(values):
    axes[1, 1].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('docs/images/tensorflow_student_classification_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*70}")
print(f"OVERALL MODEL PERFORMANCE: {test_accuracy*100:.2f}% Accuracy")
print(f"{'='*70}")

In [ ]:
# Test on sample students
test_students = [
    {
        'name': 'Excellent Student',
        'features': [95.0, 90.0, 92.0, 88.0, 95.0, 35.0, 95.0]
    },
    {
        'name': 'Very Good Student',
        'features': [88.0, 82.0, 85.0, 80.0, 88.0, 28.0, 85.0]
    },
    {
        'name': 'Good Student',
        'features': [80.0, 75.0, 78.0, 72.0, 80.0, 20.0, 75.0]
    },
    {
        'name': 'Acceptable Student',
        'features': [70.0, 60.0, 65.0, 58.0, 62.0, 15.0, 60.0]
    },
    {
        'name': 'Struggling Student',
        'features': [55.0, 45.0, 50.0, 48.0, 52.0, 10.0, 50.0]
    }
]

print("=" * 70)
print("PREDICTIONS ON SAMPLE STUDENTS (TensorFlow Model)")
print("=" * 70)

for student in test_students:
    # Create polynomial features
    features = np.array([student['features']])
    features_poly = poly.transform(features)
    features_scaled = scaler.transform(features_poly)
    
    # Make prediction
    prediction = model.predict(features_scaled, verbose=0)
    predicted_class_idx = np.argmax(prediction)
    predicted_category = category_names[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx]
    
    print(f"\n{student['name']}:")
    print(f"  Features: {student['features']}")
    print(f"  Predicted: {predicted_category} (confidence: {confidence:.4f} = {confidence*100:.2f}%)")
    print(f"  Class Probabilities:")
    for i, cat in enumerate(category_names):
        prob = prediction[0][i]
        bar = '█' * int(prob * 20)
        print(f"    {cat:15s}: {prob:.4f} ({prob*100:5.2f}%) {bar}")

print("\n" + "=" * 70)

## Step 10: Comparison with Previous Projects

In [ ]:
# Load previous project model info
project7_accuracy = 0.62  # Default
project8_accuracy = 0.85  # Default

try:
    with open('data/neural_networks/student_model_info.json', 'r') as f:
        project7_info = json.load(f)
        project7_accuracy = project7_info.get('accuracy', 0.62)
except:
    pass

try:
    # Project 8 might not have saved info yet, use default
    pass
except:
    pass

# Comparison
comparison_data = {
    'Metric': ['Framework', 'Architecture', 'Features', 'Class Balancing', 
               'Learning Rate', 'Epochs', 'Accuracy'],
    'Project 7': ['Custom NN', '7→32→16→5', '7 (original)', 'No', 
                  'Fixed 0.01', '200', f'{project7_accuracy*100:.2f}%'],
    'Project 8': ['Custom NN', 'Poly→64→128→64→32→5', '28 (polynomial)', 'Yes', 
                  'Adaptive', '500 (early stop)', f'{project8_accuracy*100:.2f}%'],
    'Project 9': ['TensorFlow/Keras', 'Poly→128→256→128→64→5', '28 (polynomial)', 'Yes + Weights', 
                  'Adam + ReduceLR', '300 (early stop)', f'{test_accuracy*100:.2f}%']
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("PROJECT 7 vs PROJECT 8 vs PROJECT 9 COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))

improvement_vs_7 = ((test_accuracy - project7_accuracy) / project7_accuracy) * 100
improvement_vs_8 = ((test_accuracy - project8_accuracy) / project8_accuracy) * 100 if project8_accuracy > 0 else 0

print(f"\n{'='*80}")
print(f"Accuracy Improvement vs Project 7: {improvement_vs_7:+.2f}%")
print(f"Absolute Improvement vs Project 7: {(test_accuracy - project7_accuracy)*100:+.2f} percentage points")
if project8_accuracy > 0:
    print(f"Accuracy Improvement vs Project 8: {improvement_vs_8:+.2f}%")
    print(f"Absolute Improvement vs Project 8: {(test_accuracy - project8_accuracy)*100:+.2f} percentage points")
print(f"{'='*80}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
projects = ['Project 7', 'Project 8', 'Project 9']
accuracies = [project7_accuracy, project8_accuracy, test_accuracy]
colors = ['lightcoral', 'lightblue', 'steelblue']

axes[0].bar(projects, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Accuracy Comparison Across Projects', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')
for i, (proj, acc) in enumerate(zip(projects, accuracies)):
    axes[0].text(i, acc + 0.02, f'{acc:.3f}', ha='center', va='bottom', 
                fontweight='bold', fontsize=11)

# Improvement visualization
improvements = [0, improvement_vs_8, improvement_vs_7]
axes[1].barh(projects, improvements, color=['gray', 'orange', 'green'], 
             edgecolor='black', linewidth=2)
axes[1].set_xlabel('Improvement vs Project 7 (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Performance Improvement', fontsize=14, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].grid(True, alpha=0.3, axis='x')
for i, (proj, imp) in enumerate(zip(projects, improvements)):
    if imp != 0:
        axes[1].text(imp + (1 if imp > 0 else -1), i, f'{imp:+.1f}%', 
                    ha='left' if imp > 0 else 'right', va='center', 
                    fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('docs/images/projects_comparison.png', dpi=150, bbox_inches='tight')
plt.show()